In [1]:
import sys
import os

# adding project root to python path
sys.path.append(os.path.abspath(".."))

In [2]:
from rag.config import DATA_RAW_DIR, VECTORSTORE_DIR, OPENAI_API_KEY

print(DATA_RAW_DIR)
print(VECTORSTORE_DIR)
print("API key loaded:", OPENAI_API_KEY is not None)

C:\Users\yshc4\Desktop\youtube-trend-analyzer\data\raw
C:\Users\yshc4\Desktop\youtube-trend-analyzer\vectorstore\chroma_db
API key loaded: True


In [3]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from rag.config import DATA_RAW_DIR
from rag.ingest import load_documents

docs = load_documents(DATA_RAW_DIR)

print("Number of documents loaded:", len(docs))

if len(docs) > 0:
    print("\nFirst document preview:")
    print(docs[0].page_content[:500])
    print("\nMetadata:")
    print(docs[0].metadata)

Number of documents loaded: 1

First document preview:
YouTube Growth Tips

Uploading consistently helps channels grow faster.
Videos between 3 to 5 minutes perform well for engagement.
Thumbnails and titles are very important for click-through rate.

Trending categories include Gaming, Music, and Entertainment.

Posting time matters:
- Best days: Friday, Saturday
- Best hours: 12 PM to 1 PM

Using relevant keywords in title and tags improves visibility.

Short videos under 1 minute can go viral quickly.

Metadata:
{'source': 'C:\\Users\\yshc4\\Desktop\\youtube-trend-analyzer\\data\\raw\\youtube_notes.txt'}


In [4]:
from rag.splitters import split_documents

chunks = split_documents(docs[:50])  # testing only first 50 docs for now
print("Number of chunks:", len(chunks))
print("\nFirst chunk preview:")
print(chunks[0].page_content[:500])
print("\nMetadata:")
print(chunks[0].metadata)

Number of chunks: 1

First chunk preview:
YouTube Growth Tips

Uploading consistently helps channels grow faster.
Videos between 3 to 5 minutes perform well for engagement.
Thumbnails and titles are very important for click-through rate.

Trending categories include Gaming, Music, and Entertainment.

Posting time matters:
- Best days: Friday, Saturday
- Best hours: 12 PM to 1 PM

Using relevant keywords in title and tags improves visibility.

Short videos under 1 minute can go viral quickly.

Metadata:
{'source': 'C:\\Users\\yshc4\\Desktop\\youtube-trend-analyzer\\data\\raw\\youtube_notes.txt'}


In [5]:
from rag.embed_store import create_vectorstore

# using only a very small sample first so we don't create a huge db yet
test_chunks = chunks[:10]

vectorstore = create_vectorstore(test_chunks)

print("Vectorstore created successfully!")
print("Test chunks stored:", len(test_chunks))

c:\Users\yshc4\Desktop\youtube-trend-analyzer\rag\embed_store.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorstore created successfully!
Test chunks stored: 1


In [6]:
from rag.embed_store import load_vectorstore
from rag.retriever import get_retriever

# load saved vector db
vectorstore = load_vectorstore()

# create retriever
retriever = get_retriever(vectorstore)

# ask a test query
query = "What are the best upload days for YouTube videos?"
results = retriever.invoke(query)

print("Number of retrieved chunks:", len(results))

if len(results) > 0:
    print("\nTop retrieved chunk:")
    print(results[0].page_content[:500])
    print("\nMetadata:")
    print(results[0].metadata)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of retrieved chunks: 3

Top retrieved chunk:
YouTube Growth Tips

Uploading consistently helps channels grow faster.
Videos between 3 to 5 minutes perform well for engagement.
Thumbnails and titles are very important for click-through rate.

Trending categories include Gaming, Music, and Entertainment.

Posting time matters:
- Best days: Friday, Saturday
- Best hours: 12 PM to 1 PM

Using relevant keywords in title and tags improves visibility.

Short videos under 1 minute can go viral quickly.

Metadata:
{'source': 'C:\\Users\\yshc4\\Desktop\\youtube-trend-analyzer\\data\\raw\\youtube_notes.txt'}


c:\Users\yshc4\Desktop\youtube-trend-analyzer\rag\embed_store.py:27: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [7]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from rag.embed_store import load_vectorstore
from rag.pipeline import answer_question

vectorstore = load_vectorstore()

question = "What are the best upload days for YouTube videos?"
answer, docs = answer_question(vectorstore, question)

print("Question:", question)
print("\nAnswer:", answer)
print("\nDocs used:", len(docs))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\yshc4\Desktop\youtube-trend-analyzer\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yshc4\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Question: What are the best upload days for YouTube videos?

Answer: Best days: Friday, Saturday

Docs used: 3
